# Train YOLO26n on PASCAL VOC with Ultralytics (Google Colab)

This notebook follows the full pipeline you requested:

1. **Prepare Colab** (GPU check, install Ultralytics)
2. **Download** the official Ultralytics VOC dataset (YOLO-compatible, auto-converted from XML)
3. **Inspect** the dataset structure, class names, label format, and what the pretrained model expects
4. **Align** VOC classes ↔ COCO classes (using the shared-class mapping you provided) so the pretrained model can be validated fairly
5. **Baseline validation** of the pretrained COCO model on the aligned VOC data
6. **Train** YOLO26n on VOC
7. **Post-training validation** to measure improvement

**Model used:** `yolo26n.pt` (current Ultralytics nano detection model pretrained on COCO).  
If you need a different checkpoint (e.g. `yolo11n.pt`), change the `MODEL` variable in the Setup cell.

**Important notes about class alignment:**
- VOC has 20 classes; COCO has 80.
- All 20 VOC classes exist in COCO (with spelling / naming differences).
- Class *indices* are different → a raw pretrained model cannot be validated directly on VOC labels.
- We therefore create a **COCO-aligned copy** of the VOC labels (remapped indices + COCO-style names) for the baseline validation.
- For actual training we use the standard `VOC.yaml` (20 classes) so the detection head is correctly sized.

## 1. Prepare Google Colab Environment

In [ ]:
# Check GPU (T4 / A100 / L4 recommended)
!nvidia-smi

import torch
print(f"\nPyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# Install / upgrade Ultralytics (and dependencies)
%pip install -U ultralytics -q

import ultralytics
ultralytics.checks()

from ultralytics import YOLO
from ultralytics.utils import SETTINGS, YAML
from pathlib import Path
import os, shutil, yaml, random
from collections import Counter
import matplotlib.pyplot as plt
from PIL import Image
import numpy as np

print("\nUltralytics ready.")

In [ ]:
# Global configuration
MODEL = "yolo26n.pt"          # change to yolo11n.pt / yolov8n.pt etc. if desired
IMGSZ = 640
EPOCHS = 50                   # increase for better final accuracy (100+ is common)
BATCH = 16                    # reduce if you hit OOM on free T4
PROJECT = "runs/voc"
NAME_BASELINE = "baseline_pretrained"
NAME_TRAIN = "train_yolo26n_voc"

# Working directories
ROOT = Path("/content")
DATASETS_DIR = Path(SETTINGS["datasets_dir"])  # usually /content/datasets or ~/.ultralytics/datasets
print(f"Datasets will live under: {DATASETS_DIR}")
print(f"Model checkpoint: {MODEL}")

## 2. Download VOC Dataset (Ultralytics-compatible)

Ultralytics ships an official `VOC.yaml` that:
- Downloads the three official VOC archives (~2.8 GB)
- Converts Pascal VOC XML → YOLO TXT labels
- Builds the standard train / val splits

We trigger the download by simply loading the dataset configuration (or by calling `model.train` / `model.val` later).  
To force the download now and inspect everything, we use the YOLO helper.

In [ ]:
# Trigger automatic download + conversion of VOC
# This may take several minutes the first time (~2.8 GB download + conversion)
from ultralytics.data.utils import check_det_dataset

data_dict = check_det_dataset("VOC.yaml")
print("\nDataset dictionary keys:", list(data_dict.keys()))
print("path:", data_dict.get("path"))
print("train:", data_dict.get("train"))
print("val:", data_dict.get("val"))
print("nc:", data_dict.get("nc"))
print("names:", data_dict.get("names"))

In [ ]:
# Locate the actual dataset root
VOC_ROOT = Path(data_dict["path"])
print("VOC root:", VOC_ROOT)
print("Exists:", VOC_ROOT.exists())

# Show directory structure (top levels)
!find {VOC_ROOT} -maxdepth 3 -type d | head -40

## 3. Inspect the Dataset & What the Model Expects

In [ ]:
# Official VOC class names (from VOC.yaml)
VOC_NAMES = {
    0: "aeroplane",
    1: "bicycle",
    2: "bird",
    3: "boat",
    4: "bottle",
    5: "bus",
    6: "car",
    7: "cat",
    8: "chair",
    9: "cow",
    10: "diningtable",
    11: "dog",
    12: "horse",
    13: "motorbike",
    14: "person",
    15: "pottedplant",
    16: "sheep",
    17: "sofa",
    18: "train",
    19: "tvmonitor",
}
print("VOC has", len(VOC_NAMES), "classes:")
for k, v in VOC_NAMES.items():
    print(f"  {k:2d}: {v}")

In [ ]:
# Load the pretrained model and inspect what *it* expects
model = YOLO(MODEL)
print(f"\nModel: {MODEL}")
print(f"Task: {model.task}")
print(f"Number of classes (nc): {model.model.nc if hasattr(model.model, 'nc') else 'N/A'}")
print("\nCOCO class names (first 20 + last few):")
names = model.names
for i in list(range(20)) + [56, 57, 58, 60, 62]:
    print(f"  {i:2d}: {names[i]}")
print("...")

In [ ]:
# Count images & labels in the official splits
def count_files(pattern):
    return len(list(VOC_ROOT.glob(pattern)))

print("Image counts:")
for split in ["train2012", "train2007", "val2012", "val2007", "test2007"]:
    n_img = count_files(f"images/{split}/*.jpg")
    n_lbl = count_files(f"labels/{split}/*.txt")
    print(f"  {split:12s}: {n_img:5d} images, {n_lbl:5d} labels")

print("\nOfficial train set (union of train*/val*): ~16 551 images")
print("Official val set (test2007):               ~ 4 952 images")

In [ ]:
# Inspect a few random label files (YOLO format: class x_center y_center width height  – all normalized)
label_files = list((VOC_ROOT / "labels" / "test2007").glob("*.txt"))
random.seed(42)
samples = random.sample(label_files, min(5, len(label_files)))

print("Sample YOLO label files (class_id + normalized xywh):\n")
for lf in samples:
    print(f"=== {lf.name} ===")
    content = lf.read_text().strip()
    if not content:
        print("  (empty – no objects)")
    else:
        for line in content.splitlines()[:6]:
            print(" ", line)
        if len(content.splitlines()) > 6:
            print("  ...")
    print()

In [ ]:
# Visualize a few images with their ground-truth boxes (using Ultralytics plot utilities)
from ultralytics.utils.plotting import Annotator, colors
import cv2

def show_gt_samples(n=4, split="test2007"):
    img_dir = VOC_ROOT / "images" / split
    lbl_dir = VOC_ROOT / "labels" / split
    imgs = list(img_dir.glob("*.jpg"))
    random.seed(0)
    chosen = random.sample(imgs, min(n, len(imgs)))

    fig, axes = plt.subplots(1, n, figsize=(5*n, 5))
    if n == 1:
        axes = [axes]

    for ax, img_path in zip(axes, chosen):
        im = cv2.imread(str(img_path))
        im = cv2.cvtColor(im, cv2.COLOR_BGR2RGB)
        h, w = im.shape[:2]
        annotator = Annotator(im.copy(), line_width=2)

        lbl_path = lbl_dir / (img_path.stem + ".txt")
        if lbl_path.exists():
            for line in lbl_path.read_text().strip().splitlines():
                cls, xc, yc, bw, bh = map(float, line.split())
                cls = int(cls)
                # denormalize
                x1 = int((xc - bw/2) * w)
                y1 = int((yc - bh/2) * h)
                x2 = int((xc + bw/2) * w)
                y2 = int((yc + bh/2) * h)
                annotator.box_label([x1, y1, x2, y2], VOC_NAMES[cls], color=colors(cls, True))

        ax.imshow(annotator.result())
        ax.set_title(img_path.name)
        ax.axis("off")

    plt.tight_layout()
    plt.show()

show_gt_samples(n=4)

## 4. Align Dataset ↔ Model (VOC ↔ COCO shared classes)

All 20 VOC classes exist in COCO.  The mapping (by meaning) is:

| VOC name      | COCO name      | COCO index |
|---------------|----------------|------------|
| aeroplane     | airplane       | 4          |
| bicycle       | bicycle        | 1          |
| bird          | bird           | 14         |
| boat          | boat           | 8          |
| bottle        | bottle         | 39         |
| bus           | bus            | 5          |
| car           | car            | 2          |
| cat           | cat            | 15         |
| chair         | chair          | 56         |
| cow           | cow            | 19         |
| diningtable   | dining table   | 60         |
| dog           | dog            | 16         |
| horse         | horse          | 17         |
| motorbike     | motorcycle     | 3          |
| person        | person         | 0          |
| pottedplant   | potted plant   | 58         |
| sheep         | sheep          | 18         |
| sofa          | couch          | 57         |
| train         | train          | 6          |
| tvmonitor     | tv             | 62         |

We create a **COCO-aligned** version of the VOC labels (and a matching YAML) so that a pretrained COCO model can be validated *without* changing its head.  This gives a true baseline of how well the pretrained weights already understand the VOC visual concepts.

In [ ]:
# VOC index → COCO index mapping
VOC_TO_COCO = {
    0: 4,   # aeroplane → airplane
    1: 1,   # bicycle → bicycle
    2: 14,  # bird → bird
    3: 8,   # boat → boat
    4: 39,  # bottle → bottle
    5: 5,   # bus → bus
    6: 2,   # car → car
    7: 15,  # cat → cat
    8: 56,  # chair → chair
    9: 19,  # cow → cow
    10: 60, # diningtable → dining table
    11: 16, # dog → dog
    12: 17, # horse → horse
    13: 3,  # motorbike → motorcycle
    14: 0,  # person → person
    15: 58, # pottedplant → potted plant
    16: 18, # sheep → sheep
    17: 57, # sofa → couch
    18: 6,  # train → train
    19: 62, # tvmonitor → tv
}

# Reverse lookup for display
COCO_NAMES_FOR_VOC = {coco_id: model.names[coco_id] for coco_id in VOC_TO_COCO.values()}
print("Aligned class names (COCO style):")
for voc_id, coco_id in sorted(VOC_TO_COCO.items()):
    print(f"  VOC {voc_id:2d} ({VOC_NAMES[voc_id]:12s}) → COCO {coco_id:2d} ({COCO_NAMES_FOR_VOC[coco_id]})")

In [ ]:
# Create a COCO-aligned copy of the labels (only for the validation split we care about)
# We keep the original VOC labels untouched for training.

ALIGNED_ROOT = VOC_ROOT.parent / "VOC_coco_aligned"
ALIGNED_LABELS = ALIGNED_ROOT / "labels" / "test2007"
ALIGNED_LABELS.mkdir(parents=True, exist_ok=True)

# Symlink / copy images (we only need the val images for baseline)
ALIGNED_IMAGES = ALIGNED_ROOT / "images" / "test2007"
if not ALIGNED_IMAGES.exists():
    ALIGNED_IMAGES.parent.mkdir(parents=True, exist_ok=True)
    # Use a symlink to save space (works on Colab)
    os.symlink(VOC_ROOT / "images" / "test2007", ALIGNED_IMAGES)
    print(f"Symlinked images → {ALIGNED_IMAGES}")

# Remap every label file
src_lbl_dir = VOC_ROOT / "labels" / "test2007"
n_converted = 0
for src in src_lbl_dir.glob("*.txt"):
    lines_out = []
    for line in src.read_text().strip().splitlines():
        parts = line.split()
        if not parts:
            continue
        voc_cls = int(parts[0])
        coco_cls = VOC_TO_COCO[voc_cls]
        lines_out.append(f"{coco_cls} {' '.join(parts[1:])}")
    (ALIGNED_LABELS / src.name).write_text("\n".join(lines_out) + ("\n" if lines_out else ""))
    n_converted += 1

print(f"Converted {n_converted} label files → {ALIGNED_LABELS}")

In [ ]:
# Write a YAML that points to the aligned data and uses the full COCO 80-class name list
# (the model still has 80 outputs; only the 20 shared classes will appear in the metrics)

aligned_yaml = {
    "path": str(ALIGNED_ROOT),
    "train": "images/test2007",   # not used for baseline val
    "val": "images/test2007",
    "test": "images/test2007",
    "names": model.names,         # full COCO names so indices match the pretrained head
}

ALIGNED_YAML_PATH = ALIGNED_ROOT / "voc_coco_aligned.yaml"
with open(ALIGNED_YAML_PATH, "w") as f:
    yaml.dump(aligned_yaml, f, sort_keys=False, default_flow_style=False)

print("Wrote aligned dataset YAML:", ALIGNED_YAML_PATH)
print(ALIGNED_YAML_PATH.read_text()[:600], "...")

## 5. Baseline Validation (Pretrained COCO model on aligned VOC)

This tells us:
- Whether the class indices now match correctly (no silent class-mismatch bugs)
- The starting mAP of the pure COCO-pretrained weights on VOC concepts (zero-shot transfer)

In [ ]:
# Baseline validation
print("Running baseline validation of pretrained model on COCO-aligned VOC val set...")
baseline_metrics = model.val(
    data=str(ALIGNED_YAML_PATH),
    imgsz=IMGSZ,
    batch=BATCH,
    project=PROJECT,
    name=NAME_BASELINE,
    exist_ok=True,
    plots=True,
    conf=0.001,          # low conf to get full PR curve
    iou=0.6,
)

print("\n===== BASELINE RESULTS (pretrained YOLO26n on VOC) =====")
print(f"mAP50-95 : {baseline_metrics.box.map:.4f}")
print(f"mAP50    : {baseline_metrics.box.map50:.4f}")
print(f"mAP75    : {baseline_metrics.box.map75:.4f}")
print("Per-class mAP50 (only classes that appear in VOC):")
# Show only the 20 relevant classes
for coco_id in sorted(VOC_TO_COCO.values()):
    # metrics.box.maps is a list indexed by class id
    if hasattr(baseline_metrics.box, "maps") and len(baseline_metrics.box.maps) > coco_id:
        print(f"  {coco_id:2d} {model.names[coco_id]:15s}: {baseline_metrics.box.maps[coco_id]:.4f}")

## 6. Train YOLO26n on the Official VOC Dataset

We now train with the **standard** `VOC.yaml` (20 classes).  
Ultralytics automatically:
- Keeps the pretrained backbone + neck weights
- Re-initializes the classification head to `nc=20`
- Fine-tunes everything end-to-end

This is the recommended transfer-learning path.

In [ ]:
# Fresh model instance for training (starts from the same pretrained weights)
model_train = YOLO(MODEL)

print(f"Starting training for {EPOCHS} epochs ...")
results = model_train.train(
    data="VOC.yaml",          # official 20-class VOC
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    project=PROJECT,
    name=NAME_TRAIN,
    exist_ok=True,
    pretrained=True,          # already true when loading a .pt
    patience=20,              # early stopping
    save=True,
    plots=True,
    # optional useful knobs for Colab:
    # cache=True,             # cache images in RAM if you have enough memory
    # workers=2,              # lower if CPU bottleneck
    # amp=True,               # mixed precision (default)
)

print("\nTraining finished.")
print("Best weights:", results.save_dir / "weights" / "best.pt")

## 7. Post-training Validation & Comparison

Validate the fine-tuned model on the same VOC val set and compare against the baseline.

In [ ]:
# Load the best checkpoint from training
best_pt = Path(PROJECT) / NAME_TRAIN / "weights" / "best.pt"
assert best_pt.exists(), f"Best weights not found at {best_pt}"

model_ft = YOLO(str(best_pt))

print("Validating fine-tuned model on official VOC val set...")
ft_metrics = model_ft.val(
    data="VOC.yaml",
    imgsz=IMGSZ,
    batch=BATCH,
    project=PROJECT,
    name="val_finetuned",
    exist_ok=True,
    plots=True,
)

print("\n===== FINE-TUNED RESULTS =====")
print(f"mAP50-95 : {ft_metrics.box.map:.4f}")
print(f"mAP50    : {ft_metrics.box.map50:.4f}")
print(f"mAP75    : {ft_metrics.box.map75:.4f}")

In [ ]:
# Side-by-side comparison
print("\n" + "="*60)
print("COMPARISON: Pretrained (COCO-aligned)  vs  Fine-tuned on VOC")
print("="*60)
print(f"{'Metric':<20} {'Baseline (COCO→VOC)':>22} {'Fine-tuned (VOC)':>20}")
print("-"*60)
print(f"{'mAP50-95':<20} {baseline_metrics.box.map:>22.4f} {ft_metrics.box.map:>20.4f}")
print(f"{'mAP50':<20} {baseline_metrics.box.map50:>22.4f} {ft_metrics.box.map50:>20.4f}")
print(f"{'mAP75':<20} {baseline_metrics.box.map75:>22.4f} {ft_metrics.box.map75:>20.4f}")
print("="*60)
print("\nNote: Baseline uses COCO class indices (80-class head).")
print("Fine-tuned uses the native 20-class VOC head.  Absolute numbers are therefore not")
print("directly comparable class-by-class, but the overall mAP improvement shows the")
print("benefit of domain / class-specific fine-tuning.")

## 8. Optional – Quick Inference Demo on a Few VOC Images

In [ ]:
# Run inference with the fine-tuned model and display a few results
demo_imgs = list((VOC_ROOT / "images" / "test2007").glob("*.jpg"))
random.seed(1)
demo_imgs = random.sample(demo_imgs, 6)

results = model_ft.predict(
    source=[str(p) for p in demo_imgs],
    imgsz=IMGSZ,
    conf=0.25,
    save=False,
    verbose=False,
)

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()
for ax, r in zip(axes, results):
    im = r.plot()  # BGR → RGB already handled by plot()
    ax.imshow(im[..., ::-1] if im.shape[-1] == 3 else im)  # safety
    ax.set_title(Path(r.path).name)
    ax.axis("off")
plt.tight_layout()
plt.show()

## 9. Summary & Next Steps

**What we did**
1. Prepared Colab + installed Ultralytics.
2. Downloaded the official YOLO-ready VOC dataset via `VOC.yaml`.
3. Inspected dataset layout, label format, and the pretrained model’s COCO class list.
4. Built an explicit VOC→COCO class-index mapping and created a COCO-aligned label set + YAML.
5. Ran a **baseline validation** of the pure pretrained weights on the aligned VOC data (confirms correct matching).
6. Fine-tuned `yolo26n` on the native 20-class VOC for the requested number of epochs.
7. Validated the fine-tuned model and compared metrics.

**Expected behaviour**
- Baseline mAP is already non-trivial because COCO and VOC share the same visual concepts.
- After fine-tuning you should see a clear lift in mAP50 / mAP50-95 on the VOC validation set.

**Useful next experiments**
- Increase `EPOCHS` (100–300) and/or use a larger model (`yolo26s.pt`, `yolo26m.pt`).
- Freeze the backbone for the first few epochs (`freeze=10`) then unfreeze.
- Export the best weights: `model_ft.export(format="onnx")` or TensorRT.
- Try the same pipeline with a different pretrained checkpoint (`yolo11n.pt`, `yolov8n.pt`, …).

All artifacts (plots, confusion matrices, best.pt, results.csv) are saved under `runs/voc/`.